# Piece markers


## Import


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from loguru import logger as lg
from rich import get_console
from rich import print as rprint
from rich.console import Console

console: Console = get_console()
console.is_jupyter = False

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from snap_fit.aruco.sheet_metadata import QRChunkHandler
from snap_fit.aruco.sheet_metadata import SheetMetadata
from snap_fit.config.aruco.metadata_zone_config import MetadataZoneConfig
from snap_fit.config.aruco.metadata_zone_config import SlotGridConfig
from snap_fit.config.aruco.sheet_aruco_config import SheetArucoConfig
from snap_fit.params.snap_fit_params import get_snap_fit_params

## Params and config


In [ ]:
project_name_params = get_snap_fit_params()
rprint(project_name_params)

## Showcase


### Step 02: SheetMetadata model


In [ ]:
meta = SheetMetadata(
    tag_name="oca",
    sheet_index=1,
    total_sheets=6,
    board_config_id="oca",
)
rprint(meta)

In [ ]:
payload = meta.to_qr_payload()
print(f"Payload: {payload!r}  ({len(payload)} bytes)")

recovered = SheetMetadata.from_qr_payload(payload)
assert recovered == meta
print("Round-trip OK")

In [ ]:
meta_unknown = SheetMetadata(
    tag_name="milano1",
    sheet_index=0,
    total_sheets=None,
    board_config_id="ab1",
)
payload_unknown = meta_unknown.to_qr_payload()
print(f"Unknown total_sheets payload: {payload_unknown!r}")
assert SheetMetadata.from_qr_payload(payload_unknown) == meta_unknown
print("Round-trip OK")

### Step 03: QRChunkHandler encode / decode


In [ ]:
handler = QRChunkHandler(n_codes=3, ecc="M")
images = handler.encode(payload)
print(f"Generated {len(images)} QR images, each {images[0].shape}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, img in zip(axes, images):
    ax.imshow(img, cmap="gray", vmin=0, vmax=255)
    ax.axis("off")
fig.suptitle("3× identical QR codes (full payload each)")
plt.tight_layout()
plt.show()

In [ ]:
for i, img in enumerate(images):
    decoded = handler.decode_first(img)
    assert decoded == payload, f"Code {i} failed: {decoded!r}"
    print(f"Code {i}: decoded OK -> {decoded!r}")

In [ ]:
blank = np.ones((100, 100), dtype=np.uint8) * 255
result = handler.decode_first(blank)
print(f"Blank image decode: {result!r}")

### Step 04: MetadataZoneConfig + SlotGridConfig


In [ ]:
zone = MetadataZoneConfig()
rprint(zone)

In [ ]:
custom_zone = MetadataZoneConfig(
    qr_n_codes=3,
    qr_ecc="M",
    text_enabled=True,
    slot_grid=SlotGridConfig(cols=10, rows=8, label_inset_px=6),
)
rprint(custom_zone)

In [ ]:
json_str = zone.model_dump_json()
rprint(json_str)
recovered_zone = MetadataZoneConfig.model_validate_json(json_str)
assert recovered_zone == zone
rprint("JSON round-trip OK")

In [ ]:
cfg_default = SheetArucoConfig.model_validate({"detector": {}})
print(f"metadata_zone default: {cfg_default.metadata_zone!r}")

cfg_with_zone = SheetArucoConfig.model_validate(
    {
        "detector": {},
        "metadata_zone": {
            "qr_n_codes": 3,
            "slot_grid": {"cols": 8, "rows": 6},
        },
    }
)
rprint(cfg_with_zone)

In [ ]:
import json
from pathlib import Path

params = get_snap_fit_params()
oca_cfg_path = params.paths.data_fol / "oca" / "oca_SheetArucoConfig.json"
if oca_cfg_path.exists():
    raw = json.loads(oca_cfg_path.read_text())
    loaded = SheetArucoConfig.model_validate(raw)
    print(f"oca config loaded OK, metadata_zone={loaded.metadata_zone!r}")
else:
    print(f"File not found (expected in dev): {oca_cfg_path}")